<a href="https://colab.research.google.com/github/mariaclarasouzadf-ops/PSP8_An-lise_Tradicional-_-Mecanismos_de_IA/blob/main/Projeto_Analise_Tradicional_vs_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Projeto Acadêmico: Avaliação Comparativa entre Análise Tradicional e Mecanismos de IA
Objetivo: Verificar ganhos preditivos e de diagnóstico multivariado em relação ao baseline descritivo.
Base: Telco Customer Churn (Kaggle / IBM Public Dataset)
"""

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split


def carregar_dados():
    url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
    dados = pd.read_csv(url)

    # Tratamento de valores nulos e tipos
    dados["TotalCharges"] = pd.to_numeric(dados["TotalCharges"], errors="coerce")
    dados.dropna(inplace=True)
    dados.drop(columns=["customerID"], inplace=True)
    return dados


def analise_tradicional(df):
    print("--- 1. ANÁLISE DESCRITIVA TRADICIONAL ---")
    taxa_media = (df["Churn"] == "Yes").mean()
    print(f"Taxa média observada de cancelamento: {taxa_media:.2%}")

    # Segmentação unidimensional simples por tipo de contrato
    tabela_contrato = (
        df.groupby("Contract")["Churn"]
        .value_counts(normalize=True)
        .unstack()["Yes"]
    )
    print("\nProporção por Tipo de Contrato:")
    print((tabela_contrato * 100).round(2).astype(str) + "%")
    return taxa_media


def executar_modelo(df):
    print("\n--- 2. MODELAGEM COM MECANISMO INTELIGENTE (MACHINE LEARNING) ---")
    df_dummies = pd.get_dummies(df, drop_first=True)
    X = df_dummies.drop(columns=["Churn_Yes"])
    y = df_dummies["Churn_Yes"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    clf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1]

    print(
        classification_report(
            y_test, y_pred, target_names=["Fidelizado", "Cancelou"]
        )
    )
    return clf, X_test, y_test, y_pred, y_prob, X.columns


def comparar_pares(y_test, y_pred, y_prob, taxa_media):
    print("\n--- 3. CONFRONTO DIRETO POR PARES ---")
    # Baseline: regra estática baseada apenas na classe majoritária da média
    pred_baseline = np.zeros_like(y_test)
    acc_tradicional = accuracy_score(y_test, pred_baseline)
    acc_modelo = accuracy_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_prob)

    print(
        f"Acurácia Método Tradicional (Regra da Média): {acc_tradicional:.2%}"
    )
    print(f"Acurácia Mecanismo de Inteligência:         {acc_modelo:.2%}")
    print(f"Capacidade de Discriminação (ROC-AUC):        {auc_score:.4f}")
    print(
        f"Diferencial de Acurácia:                     +{(acc_modelo - acc_tradicional) * 100:.2f} p.p."
    )


def exibir_relevancia(clf, colunas):
    pesos = (
        pd.Series(clf.feature_importances_, index=colunas)
        .sort_values(ascending=False)
        .head(6)
    )
    plt.figure(figsize=(8, 4))
    pesos.plot(kind="barh", color="#3b6978")
    plt.title("Variáveis com Maior Peso no Diagnóstico do Modelo")
    plt.xlabel("Grau de Importância")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    df = carregar_dados()
    taxa_media = analise_tradicional(df)
    clf, X_test, y_test, y_pred, y_prob, colunas = executar_modelo(df)
    comparar_pares(y_test, y_pred, y_prob, taxa_media)
    exibir_relevancia(clf, colunas)